# EG-VAN Phase 4C GPU Smoke Test

This notebook validates the Colab GPU environment and the plain ImageNet-pretrained EfficientNetV2S baseline. It does not train, evaluate a research run, or modify the frozen splits.

Upload the existing `EG-VAN` project folder to `MyDrive` first. Only `data/processed/images`, `data/processed/metadata_clean.csv`, and the two split CSVs are required in Colab; the raw HAM10000 data stays local.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/EG-VAN'
%cd $PROJECT_ROOT

from pathlib import Path
required = [
    'data/processed/images',
    'data/processed/metadata_clean.csv',
    'data/splits/split_naive.csv',
    'data/splits/split_leakage_aware.csv',
    'src/dataset.py',
    'src/models/baseline_effnet.py',
]
missing = [path for path in required if not (Path(PROJECT_ROOT) / path).exists()]
if missing:
    raise FileNotFoundError(f'Missing required Colab project files: {missing}')
print('Project inputs verified')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/EG-VAN
Project inputs verified


In [3]:
import random
import numpy as np
import torch
import torchvision

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('CUDA available:', torch.cuda.is_available())
print('CUDA runtime:', torch.version.cuda)
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('PHASE 4C BLOCKED: a CUDA GPU is required; CPU training is not permitted.')
if torch.__version__.split('+')[0].split('.')[0:2] != torchvision.__version__.split('+')[0].split('.')[0:2]:
    print('Version pair will be validated by model construction; major/minor versions differ.')
print('CUDA device check passed')

GPU name: Tesla T4
CUDA available: True
CUDA runtime: 12.8
PyTorch: 2.11.0+cu128
torchvision: 0.26.0+cu128
Version pair will be validated by model construction; major/minor versions differ.
CUDA device check passed


In [4]:
import sys
sys.path.insert(0, str(Path(PROJECT_ROOT) / 'src'))

from dataset import CLASS_NAMES, HAM10000Dataset, validate_split_integrity
from models.baseline_effnet import build_model
from train import focal_loss, make_optimizer_and_scheduler, make_transforms
from torch.utils.data import DataLoader

naive_stats = validate_split_integrity(Path(PROJECT_ROOT) / 'data/splits/split_naive.csv', require_lesion_isolation=False)
leakage_stats = validate_split_integrity(Path(PROJECT_ROOT) / 'data/splits/split_leakage_aware.csv', require_lesion_isolation=True)
print('Naive split integrity:', naive_stats)
print('Leakage-aware split integrity:', leakage_stats)

model, weights = build_model(pretrained=True)
assert weights is not None
assert model.classifier[-1].out_features == 7
print('EfficientNetV2S loaded with ImageNet weights:', weights)
print('Classifier output classes:', model.classifier[-1].out_features)

Naive split integrity: {'rows': 10015, 'unique_lesions': 7470, 'crossing_lesions': 764}
Leakage-aware split integrity: {'rows': 10015, 'unique_lesions': 7470, 'crossing_lesions': 0}
EfficientNetV2S loaded with ImageNet weights: EfficientNet_V2_S_Weights.IMAGENET1K_V1
Classifier output classes: 7


In [5]:
train_transform, eval_transform = make_transforms(weights)
split_csv = Path(PROJECT_ROOT) / 'data/splits/split_naive.csv'
images_dir = Path(PROJECT_ROOT) / 'data/processed/images'
dataset = HAM10000Dataset(images_dir, split_csv, 'train', train_transform)
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
images, labels = next(iter(loader))
print('Batch images:', tuple(images.shape))
print('Batch labels:', tuple(labels.shape))
assert images.shape[0] == 2 and images.shape[1:] == (3, 384, 384)
assert labels.min().item() >= 0 and labels.max().item() < 7
print('Dataset batch and transform check passed')

Batch images: (2, 3, 384, 384)
Batch labels: (2,)
Dataset batch and transform check passed


In [6]:
device = torch.device('cuda')
model = model.to(device).train()
images = images.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True)
optimizer, scheduler = make_optimizer_and_scheduler(model)
scaler = torch.amp.GradScaler('cuda', enabled=True)
optimizer.zero_grad(set_to_none=True)
with torch.amp.autocast('cuda', enabled=True):
    logits = model(images)
    loss = focal_loss(logits, labels)
assert tuple(logits.shape) == (2, 7)
assert torch.isfinite(loss).item()
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
scheduler.step(float(loss.detach().cpu()))
print('Forward pass: PASS')
print('Output shape:', tuple(logits.shape))
print('Focal loss:', float(loss.detach().cpu()))
print('Backward pass: PASS')
print('Optimizer step: PASS')
print('Mixed precision: PASS')

Forward pass: PASS
Output shape: (2, 7)
Focal loss: 0.3575671315193176
Backward pass: PASS
Optimizer step: PASS
Mixed precision: PASS


## Smoke-test gate

If every code cell above passes, record the printed environment and checks as the Phase 4C GPU smoke-test result. Do not start either 25-epoch experiment from this notebook.